# Week 2, Day 3: Broadcasting and vectorised thinking

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w002/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
rates = rng.gamma(2.0, 5.0, size=(20, 60))       # (neurons, trials), Hz

m = rates.mean(axis=1)                            # (20,)
try:
    rates - m
except ValueError as e:
    print("error:", e)

m_keep = rates.mean(axis=1, keepdims=True)        # (20, 1)
centred = rates - m_keep                          # (20, 60)
print(centred.shape, np.allclose(centred.mean(axis=1), 0))

centred2 = rates - m[:, None]                     # same thing with None
print(np.allclose(centred, centred2))

In [ ]:
import matplotlib.pyplot as plt
n = 40
pref = np.linspace(0, np.pi, n, endpoint=False)[:, None]   # (40, 1)
theta = np.linspace(0, np.pi, 36, endpoint=False)          # (36,)
r0, g, kappa = 2.0, 30.0, 2.5
tuning = r0 + g * np.exp(kappa * (np.cos(2 * (theta - pref)) - 1))   # (40, 36)
print(tuning.shape)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
for i in [0, 10, 20, 30]:
    axes[0].plot(np.degrees(theta), tuning[i], label=f"pref {np.degrees(pref[i, 0]):.0f}°")
axes[0].set_xlabel("stimulus orientation (deg)"); axes[0].set_ylabel("rate (Hz)"); axes[0].legend(fontsize=7)
axes[1].imshow(tuning, aspect="auto", extent=[0, 180, n, 0], cmap="magma")
axes[1].set_xlabel("stimulus orientation (deg)"); axes[1].set_ylabel("neuron (sorted by preference)")
plt.tight_layout(); plt.show()

### Exercise: Z-score each neuron

Z-score `rates` (shape neurons by trials) so that each neuron has mean 0 and standard deviation 1 across trials. Store it in `z`. No loops.

In [ ]:
z = ...
print(z.shape, z.mean(axis=1)[:3].round(3), z.std(axis=1)[:3].round(3))

In [ ]:
import numpy as np
def test_shape():
    "z has the same shape as rates"
    assert z.shape == rates.shape

def test_standardised():
    "each neuron has mean 0 and SD 1 across trials"
    assert np.allclose(z.mean(axis=1), 0) and np.allclose(z.std(axis=1), 1)

test_shape()
test_standardised()
print("All checks passed")

### Exercise: Simulate trials from tuning curves

Simulate 10 repeats of each stimulus with Poisson spike counts in a 0.2 s window: the mean count is `tuning * 0.2`. Use broadcasting so `counts` has shape (40, 36, 10). Hint: `rng.poisson(lam, size=...)` broadcasts `lam` if you give it a trailing axis.

In [ ]:
rng = np.random.default_rng(1)
lam = ...
counts = ...
print(counts.shape)

In [ ]:
import numpy as np
def test_shape():
    "counts has shape (neurons, stimuli, repeats)"
    assert counts.shape == (40, 36, 10)

def test_mean():
    "average count matches tuning times window"
    est = counts.mean(axis=2) / 0.2
    assert np.abs(est - tuning).mean() < 3.0, "Mean counts / 0.2 s should be close to the tuning curves."

test_shape()
test_mean()
print("All checks passed")

### Exercise: All pairwise distances

Write `pairwise(P)`: given points `P` of shape (n, d), return the (n, n) matrix of Euclidean distances. Use broadcasting: `P[:, None, :] - P[None, :, :]` has shape (n, n, d). This is how representational distance matrices are built in Week 117.

In [ ]:
def pairwise(P):
    ...

P = np.array([[0.0, 0.0], [3.0, 4.0], [6.0, 8.0]])
print(pairwise(P))

In [ ]:
import numpy as np
def test_dist():
    "D[i, j] is the Euclidean distance between rows i and j"
    P = np.random.default_rng(9).normal(size=(7, 3))
    expect = np.array([[np.linalg.norm(a - b) for b in P] for a in P])
    assert np.allclose(pairwise(P), expect)

test_dist()
print("All checks passed")